# Microproyecto 2 - Analisis de sentimientos: procesamiento de lenguaje natural

### Ricardo Jose Garzon, Luisa Fernanda Aristizabal y Maria Fernanda Hurtado

**Deep Learning - 2026. Semestre 2**

### Introduccion 

aqui va la introduccion 

### 1. Importaciones

In [1]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [2]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import re
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split


### 2. cargado, analisis y preprocesamiento de datos

In [3]:
df = pd.read_csv("movie.csv")
df.head()


,text,label
0,I grew up (b. 1965) watching and loving the Th...,0
1,"When I put this movie in my DVD player, and sa...",0
2,Why do people who do not know what a particula...,0
3,Even though I have great interest in Biblical ...,0
4,Im a die hard Dads Army fan and nothing will e...,1


Con este info podemos observar 1. que no existen valores nulos, el dataset esta "limpio" estructuralmente, ya tenemos la variable objetivo lista

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    40000 non-null  str  
 1   label   40000 non-null  int64
dtypes: int64(1), str(1)
memory usage: 625.1 KB


In [5]:
df['label'].value_counts()

label
0    20019
1    19981
Name: count, dtype: int64

**Revisión de duplicados**

El dataset IMDB suele contener reseñas repetidas. Si quedan en train y test contaminan la evaluación, así que las eliminamos antes de cualquier análisis posterior.

In [6]:
print("Duplicados antes:", df.duplicated(subset="text").sum())
df = df.drop_duplicates(subset="text").reset_index(drop=True)
print("Filas tras eliminar duplicados:", len(df))
print(df["label"].value_counts())

Duplicados antes: 277
Filas tras eliminar duplicados: 39723
label
1    19908
0    19815
Name: count, dtype: int64


In [7]:
# Longitud de las reseñas
df['length'] = df['text'].apply(lambda x: len(x.split()))

# Revisar percentiles
print(df['length'].describe())
print(df['length'].quantile([0.50, 0.75, 0.90, 0.95]))

count    39723.000000
mean       231.486142
std        171.367657
min          4.000000
25%        126.000000
50%        173.000000
75%        282.000000
max       2470.000000
Name: length, dtype: float64
0.50    173.0
0.75    282.0
0.90    453.0
0.95    590.0
Name: length, dtype: float64


Con esto podemos observar diferentes comportamientos 1. hay mucha variabilidad, ya que algunas resenas pueden ser muy cortas (4 palabras) otras son demasiado largas (2470). La media es de 231, la mediana es de 173, como la media es mayor, podemos concluir que hay outliers largos (resenas enormes). Ahora 50% de reseñas ≤ 173 palabras 75% ≤ 282 palabras. 

**Se definió una longitud máxima de secuencia de 300 palabras para el modelo, valor que cubre aproximadamente el percentil 80 de la distribución de longitudes. Esto permite capturar la mayor parte del contenido relevante sin incrementar innecesariamente el costo computacional. Las reseñas más largas son truncadas, mientras que las más cortas son rellenadas mediante padding.**

El uso de un tamaño de secuencia fijo es necesario debido a las restricciones de entrada de las redes neuronales, particularmente en modelos recurrentes como LSTM, que requieren entradas de dimensión uniforme.


In [8]:
max_len =  300
max_words = 10000

**LIMPIEZA DE TEXTO**

In [9]:
def clean_text(text):
    text = BeautifulSoup(text, "html.parser").get_text()
    text = text.lower()
    text = re.sub(r"[^a-zA-Z']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)

df[["text", "clean_text", "label"]].head()

,text,clean_text,label
0,I grew up (b. 1965) watching and loving the Th...,i grew up b watching and loving the thunderbir...,0
1,"When I put this movie in my DVD player, and sa...",when i put this movie in my dvd player and sat...,0
2,Why do people who do not know what a particula...,why do people who do not know what a particula...,0
3,Even though I have great interest in Biblical ...,even though i have great interest in biblical ...,0
4,Im a die hard Dads Army fan and nothing will e...,im a die hard dads army fan and nothing will e...,1


**Verificación de reseñas vacías tras la limpieza**

El regex puede dejar strings vacíos si una reseña original contenía solo HTML o símbolos. Las eliminamos para evitar entradas vacías al tokenizador.

In [40]:
vacios = (df["clean_text"].str.strip() == "").sum()
print("Reseñas vacías tras limpieza:", vacios)

df = df[df["clean_text"].str.strip() != ""].reset_index(drop=True)
print("Filas finales:", len(df))

Reseñas vacías tras limpieza: 0
Filas finales: 39723


In [41]:
X_text = df["clean_text"]
y = df["label"].values

In [42]:
X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    X_text,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print(X_train_text.shape)
print(X_val_text.shape)

(27806,)
(5958,)


**Tokenización y padding**

El tokenizador se ajusta **únicamente con el conjunto de entrenamiento** para evitar fugas de información (data leakage). Se reserva un token `<OOV>` para palabras no vistas que aparezcan en validación o test.

In [43]:
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_val_seq   = tokenizer.texts_to_sequences(X_val_text)
X_test_seq  = tokenizer.texts_to_sequences(X_test_text)

print("Ejemplo secuencia (primeros 20 tokens):", X_train_seq[0][:20])

Ejemplo secuencia (primeros 20 tokens): [68, 10, 37, 4891, 1, 59, 7, 4, 309, 759, 3, 32, 567, 4667, 3, 8805, 1730, 40, 112, 8]


**Padding a longitud fija (`max_len = 300`)**

Las reseñas más cortas se rellenan con ceros y las más largas se truncan, ambas operaciones al final de la secuencia (`post`).

In [44]:
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post", truncating="post")
X_val_pad   = pad_sequences(X_val_seq,   maxlen=max_len, padding="post", truncating="post")
X_test_pad  = pad_sequences(X_test_seq,  maxlen=max_len, padding="post", truncating="post")

print("Shape train:", X_train_pad.shape)
print("Shape val:  ", X_val_pad.shape)
print("Shape test: ", X_test_pad.shape)

Shape train: (27806, 300)
Shape val:   (5958, 300)
Shape test:  (5959, 300)


**Tamaño del vocabulario**

Valor que se usará en la capa `Embedding` del modelo RNN.

In [45]:
vocab_size = min(max_words, len(tokenizer.word_index) + 1)
print("vocab_size:", vocab_size)
print("Total palabras únicas vistas en train:", len(tokenizer.word_index))

vocab_size: 10000
Total palabras únicas vistas en train: 93559


 **Pipeline de datos**

Se construye un pipeline que entregará los datos al modelo RNN por lotes (batches), con `shuffle` solo en entrenamiento y `prefetch` para solapar I/O y cómputo. Validación y test no se mezclan para que las métricas sean reproducibles.

In [46]:
import tensorflow as tf

BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

train_pipeline = (
    tf.data.Dataset.from_tensor_slices((X_train_pad, y_train))
    .shuffle(buffer_size=len(X_train_pad), seed=42)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_pipeline = (
    tf.data.Dataset.from_tensor_slices((X_val_pad, y_val))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_pipeline = (
    tf.data.Dataset.from_tensor_slices((X_test_pad, y_test))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("train_pipeline:", train_pipeline)
print("val_pipeline:  ", val_pipeline)
print("test_pipeline: ", test_pipeline)

train_pipeline: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 300), dtype=tf.int32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>
val_pipeline:   <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 300), dtype=tf.int32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>
test_pipeline:  <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 300), dtype=tf.int32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>


**Verificación rápida del pipeline**

Tomamos un batch para confirmar shapes y dtypes antes de pasarlo al modelo.

In [47]:
for x_batch, y_batch in train_pipeline.take(1):
    print("X batch shape:", x_batch.shape, "dtype:", x_batch.dtype)
    print("y batch shape:", y_batch.shape, "dtype:", y_batch.dtype)

print("Batches por epoch (train):", len(train_pipeline))
print("Batches por epoch (val):  ", len(val_pipeline))
print("Batches por epoch (test): ", len(test_pipeline))

X batch shape: (64, 300) dtype: <dtype: 'int32'>
y batch shape: (64,) dtype: <dtype: 'int64'>
Batches por epoch (train): 435
Batches por epoch (val):   94
Batches por epoch (test):  94
